# House Sale Price Prediction - Improved

This notebook builds on `house-sale-price.ipynb` with changes aimed at reducing validation/leaderboard error:

- Log-transform the skewed target (`SalePrice`)
- Remove known Ames-housing outliers
- Ordinal-encode quality columns instead of one-hot (they have a natural order)
- Add engineered features (total area, house age, total baths, has-X flags)
- Median imputation for meaningfully-missing numeric fields (0 = "doesn't have it")
- Train/validation split done **before** imputation/scaling/encoding (no leakage)
- Hyperparameter tuning (RandomizedSearchCV) for Random Forest and XGBoost
- A simple ensemble (blend) of the best models
- Final refit on 100% of the training data before predicting the Kaggle test set


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
train_df.shape, test_df.shape


In [ ]:
train_df.dropna(subset=['SalePrice'], inplace=True)
train_df.shape


### Remove known outliers

A small number of very large houses (`GrLivArea` > 4000) sold for unusually low prices — a well-known data quirk in the Ames housing dataset that hurts every model if left in.


In [ ]:
train_df = train_df[~((train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000))].reset_index(drop=True)
train_df.shape


### Drop columns that are mostly missing

Computed from the training data only, then applied to both train and test so both see the same feature set.


In [ ]:
high_na_cols = [col for col in train_df.columns if train_df[col].isna().sum() > len(train_df) * 0.5]
high_na_cols


In [ ]:
train_df.drop(columns=high_na_cols, inplace=True)
test_df.drop(columns=[c for c in high_na_cols if c in test_df.columns], inplace=True)
train_df.shape, test_df.shape


## Split Inputs / Target

In [ ]:
test_ids = test_df['Id']

train_in = train_df.drop('SalePrice', axis=1)
train_out = np.log1p(train_df['SalePrice'])  # log-transform: SalePrice is right-skewed
test_in = test_df.copy()

train_in.shape, train_out.shape, test_in.shape


## Ordinal-Encode Quality Columns

Columns like `ExterQual`/`KitchenQual` are categorical strings but have a real order (Po < Fa < TA < Gd < Ex). One-hot encoding throws that order away; mapping to integers keeps it.


In [ ]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
qual_cols = [c for c in qual_cols if c in train_in.columns]

def encode_quality(df):
    df = df.copy()
    for col in qual_cols:
        df[col] = df[col].map(qual_map).fillna(0)
    return df

train_in = encode_quality(train_in)
test_in = encode_quality(test_in)


## Fill "Meaningful" Missing Numeric Values

For several numeric columns, NaN doesn't mean "unknown" — it means "the house doesn't have this" (no garage, no basement, no masonry veneer). Mean-imputing those is misleading; 0 is correct.


In [ ]:
zero_fill_cols = ['GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                   'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
zero_fill_cols = [c for c in zero_fill_cols if c in train_in.columns]

train_in[zero_fill_cols] = train_in[zero_fill_cols].fillna(0)
test_in[zero_fill_cols] = test_in[zero_fill_cols].fillna(0)


## Feature Engineering

In [ ]:
def add_features(df):
    df = df.copy()
    df['TotalSF'] = df[['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']].fillna(0).sum(axis=1)
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalBath'] = (df['FullBath'].fillna(0) + 0.5 * df['HalfBath'].fillna(0) +
                        df['BsmtFullBath'].fillna(0) + 0.5 * df['BsmtHalfBath'].fillna(0))
    df['HasPool'] = (df['PoolArea'].fillna(0) > 0).astype(int) if 'PoolArea' in df.columns else 0
    df['HasGarage'] = (df['GarageArea'].fillna(0) > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'].fillna(0) > 0).astype(int)
    df['Has2ndFloor'] = (df['2ndFlrSF'].fillna(0) > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'].fillna(0) > 0).astype(int)
    return df

train_in = add_features(train_in)
test_in = add_features(test_in)
train_in.shape, test_in.shape


## Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = train_in.select_dtypes(include=np.number).columns
categorical_cols = train_in.select_dtypes(exclude=np.number).columns

if 'Id' in numeric_cols:
    numeric_cols = numeric_cols.drop('Id')

numeric_cols = numeric_cols.to_list()
categorical_cols = categorical_cols.to_list()
len(numeric_cols), len(categorical_cols)


## Train / Validation Split

Done **before** imputation, scaling, and encoding, so nothing about the validation rows leaks into any fitted transformer.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_in,
    train_out,
    test_size=0.2,
    random_state=42
  )

X_train.shape, X_val.shape, y_train.shape, y_val.shape


## Missing Values - Numeric

Median is more robust to skewed numeric columns than mean.


In [ ]:
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train[numeric_cols])


In [ ]:
X_train[numeric_cols] = imputer.transform(X_train[numeric_cols])
X_val[numeric_cols] = imputer.transform(X_val[numeric_cols])


## Feature Scaling - MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
scaler.fit(X_train[numeric_cols])


In [ ]:
X_train[numeric_cols] = scaler.transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])


## Missing Values - Categoric

In [ ]:
X_train[categorical_cols] = X_train[categorical_cols].fillna('None')
X_val[categorical_cols] = X_val[categorical_cols].fillna('None')


## One Hot Encoding - Categoric

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[categorical_cols])


In [ ]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_cols]),
    columns=encoded_cols,
    index=X_train.index
)

val_encoded = pd.DataFrame(
    encoder.transform(X_val[categorical_cols]),
    columns=encoded_cols,
    index=X_val.index
)


In [ ]:
feature_cols = numeric_cols + encoded_cols

X_train = pd.concat([X_train, train_encoded], axis=1)[feature_cols]
X_val = pd.concat([X_val, val_encoded], axis=1)[feature_cols]

X_train.shape, X_val.shape


## Model Evaluation

In [ ]:
results = []


In [ ]:
def rmse(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()
    return np.sqrt(np.mean(np.square(t_vals - p_vals)))


In [ ]:
def mae(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()
    return np.mean(np.abs(t_vals - p_vals))


`evaluate` reports error in both log-space (what Kaggle actually scores this competition on) and back-transformed dollars (easier to read).


In [ ]:
def evaluate(model_name, targets_log, predictions_log):
    rmse_log = rmse(targets_log, predictions_log)
    mae_log = mae(targets_log, predictions_log)

    targets_dollars = np.expm1(targets_log)
    predictions_dollars = np.expm1(predictions_log)
    rmse_dollars = rmse(targets_dollars, predictions_dollars)
    mae_dollars = mae(targets_dollars, predictions_dollars)

    results.append({
        "model": model_name,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "rmse": rmse_dollars,
        "mae": mae_dollars,
    })

    print(f"{model_name}: RMSE(log)={rmse_log:.5f}  RMSE=${rmse_dollars:,.0f}  MAE=${mae_dollars:,.0f}")


## Model Building

### Ridge Regression

With ~250+ features and ~1200 rows, plain `LinearRegression` overfits badly. Ridge adds L2 regularization; `alpha` is tuned with a small search.


In [ ]:
ridge_params = {'alpha': [0.1, 1, 5, 10, 20, 50, 100]}

ridge_search = RandomizedSearchCV(
    Ridge(random_state=42),
    ridge_params,
    n_iter=7,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
)
ridge_search.fit(X_train, y_train)
ridge_model = ridge_search.best_estimator_
ridge_search.best_params_


In [ ]:
ridge_preds = ridge_model.predict(X_val)
evaluate("Ridge Regression", y_val, ridge_preds)


### Random Forest Regressor (tuned)

In [ ]:
rf_params = {
    'n_estimators': [200, 400, 600, 800],
    'max_depth': [None, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5, 1.0],
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    n_iter=20,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_
rf_search.best_params_


In [ ]:
rf_preds = rf_model.predict(X_val)
evaluate("Random Forest Regressor (tuned)", y_val, rf_preds)


### XGBoost (tuned)

In [ ]:
xgb_params = {
    'n_estimators': [300, 500, 800, 1000],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_params,
    n_iter=25,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
xgb_model = xgb_search.best_estimator_
xgb_search.best_params_


In [ ]:
xgb_preds = xgb_model.predict(X_val)
evaluate("XGBoost (tuned)", y_val, xgb_preds)


### Ensemble (blend)

Averaging predictions from two different model families usually beats either alone.


In [ ]:
ensemble_preds = 0.5 * rf_preds + 0.5 * xgb_preds
evaluate("Ensemble (RF + XGB)", y_val, ensemble_preds)


### Compare Models

In [ ]:
pd.DataFrame(results).sort_values('rmse')


## Final Refit on Full Training Data

Model selection and tuning above used a held-out validation split. For the actual Kaggle submission, refit the chosen approach (RF + XGB ensemble) on **all** available labeled rows, using fresh transformers fit on the full training set (no leakage concern here — the test set is a separate, disjoint set of houses).


In [ ]:
full_imputer = SimpleImputer(strategy='median')
full_imputer.fit(train_in[numeric_cols])

train_in[numeric_cols] = full_imputer.transform(train_in[numeric_cols])
test_in[numeric_cols] = full_imputer.transform(test_in[numeric_cols])


In [ ]:
full_scaler = MinMaxScaler()
full_scaler.fit(train_in[numeric_cols])

train_in[numeric_cols] = full_scaler.transform(train_in[numeric_cols])
test_in[numeric_cols] = full_scaler.transform(test_in[numeric_cols])


In [ ]:
train_in[categorical_cols] = train_in[categorical_cols].fillna('None')
test_in[categorical_cols] = test_in[categorical_cols].fillna('None')


In [ ]:
full_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
full_encoder.fit(train_in[categorical_cols])
full_encoded_cols = list(full_encoder.get_feature_names_out(categorical_cols))

train_full_encoded = pd.DataFrame(
    full_encoder.transform(train_in[categorical_cols]),
    columns=full_encoded_cols,
    index=train_in.index
)

test_full_encoded = pd.DataFrame(
    full_encoder.transform(test_in[categorical_cols]),
    columns=full_encoded_cols,
    index=test_in.index
)


In [ ]:
full_feature_cols = numeric_cols + full_encoded_cols

X_train_full = pd.concat([train_in, train_full_encoded], axis=1)[full_feature_cols]
X_test = pd.concat([test_in, test_full_encoded], axis=1)[full_feature_cols]
y_train_full = train_out

X_train_full.shape, X_test.shape


In [ ]:
final_rf = RandomForestRegressor(**rf_search.best_params_, random_state=42)
final_rf.fit(X_train_full, y_train_full)

final_xgb = XGBRegressor(**xgb_search.best_params_, random_state=42)
final_xgb.fit(X_train_full, y_train_full)


## Kaggle Submission

In [ ]:
test_preds_log = 0.5 * final_rf.predict(X_test) + 0.5 * final_xgb.predict(X_test)
test_preds = np.expm1(test_preds_log)


In [ ]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_preds
})
submission.head()


In [ ]:
submission.to_csv('submission_v2.csv', index=False)
